# Convolution 2.0

## Import libraries

In [1]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as no
import pandas as pd
from pathlib import Path
import pypher

## Set directories

Determine file paths and obtain lists of galaxy images and PSF files for each survey inside Input/

In [6]:
CWD = Path.cwd()
ROOT = CWD.parents[1]
ASTROVELLO_DIR = ROOT / "AsTrovello_2_0"

input_dir = ASTROVELLO_DIR / "Input"

survey_paths = list(input_dir.glob("*"))
survey_names = [f.name for f in survey_paths]

galaxy = "ngc1097"

survey_names

['PHANGS', 'S4G']

## Obtain all survey files (Science images and PSFs)

In [11]:
image_files = []
psf_files = []
for survey in survey_names:
    image_dir = input_dir / survey / "galaxies" / galaxy
    psf_dir = input_dir / survey / "PSF"

    if survey == "PHANGS":
        current_image_files = list(image_dir.glob("*_exp-drc-sci.fits"))
        current_psf_files = list(psf_dir.glob("*PSFSTD*.fits"))
    elif survey == "S4G":
        current_image_files = list(image_dir.glob(f"{galaxy.upper()}.phot.*.fits"))
        current_psf_files = list(psf_dir.glob("*_col129_row129.fits"))

    image_files = image_files + current_image_files
    psf_files = psf_files + current_psf_files


In [14]:
image_files

[WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/galaxies/ngc1097/hlsp_phangs-hst_hst_wfc3-uvis_ngc1097mosaic_f555w_v1_exp-drc-sci.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/PHANGS/galaxies/ngc1097/hlsp_phangs-hst_hst_wfc3-uvis_ngc1097mosaic_f814w_v1_exp-drc-sci.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/galaxies/ngc1097/NGC1097.phot.1.fits'),
 WindowsPath('c:/Users/dedet/Desktop/AsTrovello_2_0/Input/S4G/galaxies/ngc1097/NGC1097.phot.2.fits')]

## Determine PSF resolutions 
Calculate FWHM to determine each surveys resolution and define the master file for convolution (lowest resolution).

### 1. Return average PSF in PHANGS files (regular files have many dimensions)

In [21]:
phangs_img_header = fits.getheader(image_files[2], 0) 
phangs_img_header["INSTRUME"]

'IRAC'

In [ ]:
SURVEY_CONFIG = {
                    "PHANGS": 
                    {
                        "TELESCOP": "HST",
                        "INSTRUME": "WFC3",
                        "pixel_scale_arcsec": 0.0395,
                        "binned_factor": 4,
                        "unit_type": "electrons/s", # usado em units.py
                        "force_tan_sip": False
                    },
                    "S4G":
                    {
                        "TELESCOP": "Spitzer",
                        "INSTRUME": "IRAC",
                        "pixel_scale_arcsec": 
                        {
                            1: 1.221, # Channel 1
                            2: 1.223 # Channel 2
                        },
                        "binned_factor": 5,
                        "unit_type": "mjy/sr", # usado em units.py
                        "force_tan_sip": True
                    }
                }